# 03 — Historical LOB Feature Engineering

## V2 research question

Do summary features derived from the 50 strictly pre-submission limit-order-book snapshots improve fixed-horizon fill-probability prediction beyond the static order, LOB, and recent-flow variables available at `t0`?

This notebook builds and validates the historical feature layer needed to answer that question. It does **not** fit models, tune thresholds, inspect test outcomes, or modify the raw files.

### Scope

- Stream the JSON history so that all 500,000 snapshots are not expanded in memory at once.
- Construct the seven feature families required by the project brief.
- Verify order IDs, sequence length, chronology, and one-to-one joining.
- Add explicitly documented buy/sell orientation for directional variables.
- Inspect feature quality using train and validation metadata only; keep test outcomes sealed.

In [1]:
from collections import Counter
from pathlib import Path
import gzip
import os
import time

import ijson
import numpy as np
import pandas as pd

RANDOM_SEED = 42
HORIZONS = (5, 30, 60)
TICK_SIZE = 0.01

project_root_override = os.environ.get("LIMIT_ORDER_PROJECT_ROOT")
PROJECT_ROOT = Path(project_root_override) if project_root_override else Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

excel_files = sorted(RAW_DIR.glob("*.xlsx"))
json_files = sorted(
    list(RAW_DIR.glob("*.json"))
    + list(RAW_DIR.glob("*.json.gz"))
)

assert len(excel_files) == 1, f"Expected one Excel file, found {len(excel_files)}"
assert len(json_files) == 1, f"Expected one JSON file, found {len(json_files)}"

EXCEL_PATH = excel_files[0]
JSON_PATH = json_files[0]

print("Project root:", PROJECT_ROOT)
print("Excel source:", EXCEL_PATH.name)
print("JSON source:", JSON_PATH.name)
print("Random seed reserved for downstream modeling:", RANDOM_SEED)

Project root: /Users/yaya/Documents/GitHub/limit-order-fill-prediction
Excel source: AI_Order_Execution.xlsx
JSON source: Single_Symbol_LOB_History.json
Random seed reserved for downstream modeling: 42


In [2]:
orders = pd.read_excel(EXCEL_PATH, sheet_name="Orders")

required_columns = {
    "order_id",
    "symbol",
    "side",
    "data_split",
    "trading_day",
    "observation_timestamp",
}
missing_required = required_columns - set(orders.columns)
assert not missing_required, f"Missing required columns: {sorted(missing_required)}"

assert len(orders) == 10_000
assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique
assert orders["symbol"].nunique() == 1
assert orders["symbol"].iloc[0] == "AAPL"

split_counts = (
    orders["data_split"]
    .astype(str)
    .str.lower()
    .value_counts()
    .reindex(["train", "validation", "test"])
)
assert split_counts.to_dict() == {
    "train": 7000,
    "validation": 1500,
    "test": 1500,
}

print("Orders shape:", orders.shape)
print("Symbol:", orders["symbol"].unique().tolist())
display(split_counts.rename("rows").to_frame())

Orders shape: (10000, 72)
Symbol: ['AAPL']


,rows
data_split,
train,7000
validation,1500
test,1500


## Feature definitions

The constructed variables follow the definitions in `docs/feature_reference.md`:

1. **Price movement:** midpoint return, best-quote change counts, and net quote direction.
2. **Spread behavior:** mean, maximum, final spread, and one-tick fraction.
3. **Imbalance behavior:** mean, standard deviation, range, final value, and linear trend.
4. **Depth change:** bid and ask depth changes, followed by same/opposite-side orientation.
5. **Microprice signal:** final gap from midpoint, mean gap, and trend.
6. **Event composition:** event-type counts and type-by-side counts.
7. **Event timing:** lookback span, mean inter-event time, and final-event distance to submission.

All JSON-derived predictors use only snapshots strictly earlier than the original order submission. `order_type` is intentionally excluded because it describes the synthetic generation mechanism.

In [3]:
EVENT_TYPES = (
    "limit_submission",
    "cancellation",
    "market_trade",
    "modification",
)
EVENT_SIDES = ("bid", "ask")


def open_json_binary(path):
    if path.name.endswith(".json.gz"):
        return gzip.open(path, mode="rb")
    return path.open(mode="rb")


def linear_trend(values):
    # OLS slope against sequence steps 1..n using a stable centered formula.
    y = np.asarray(values, dtype=float)
    x = np.arange(1, len(y) + 1, dtype=float)
    x_centered = x - x.mean()
    return float(np.dot(x_centered, y - y.mean()) / np.dot(x_centered, x_centered))


def summarize_history(order):
    history = order.get("history", [])
    sequence_steps = [snapshot.get("sequence_step") for snapshot in history]
    offsets_ms = np.asarray(
        [float(snapshot["milliseconds_before_submission"]) for snapshot in history],
        dtype=float,
    )

    submission_time = pd.Timestamp(order["submission_timestamp"])
    event_times = pd.DatetimeIndex(
        [pd.Timestamp(snapshot["event_timestamp"]) for snapshot in history]
    )

    mid = np.asarray([float(snapshot["mid_price"]) for snapshot in history])
    spread = np.asarray([float(snapshot["spread_ticks"]) for snapshot in history])
    imbalance = np.asarray(
        [float(snapshot["five_level_bid_ask_imbalance"]) for snapshot in history]
    )
    microprice = np.asarray(
        [float(snapshot["microprice"]) for snapshot in history]
    )
    micro_gap_bps = 10_000 * (microprice - mid) / mid

    bid_l1 = np.asarray([float(snapshot["bid_prices"][0]) for snapshot in history])
    ask_l1 = np.asarray([float(snapshot["ask_prices"][0]) for snapshot in history])
    bid_depth = np.asarray(
        [sum(float(value) for value in snapshot["bid_volumes"]) for snapshot in history]
    )
    ask_depth = np.asarray(
        [sum(float(value) for value in snapshot["ask_volumes"]) for snapshot in history]
    )

    event_type_counts = Counter(snapshot["event_type"] for snapshot in history)
    event_type_side_counts = Counter(
        (snapshot["event_type"], snapshot["event_side"])
        for snapshot in history
    )

    row = {
        "order_id": str(order["order_id"]),
        "history_snapshot_count": len(history),
        "check_sequence_1_to_50": sequence_steps == list(range(1, 51)),
        "check_offsets_strictly_negative": bool(np.all(offsets_ms < 0)),
        "check_offsets_strictly_increasing": bool(np.all(np.diff(offsets_ms) > 0)),
        "check_event_timestamps_before_submission": bool(np.all(event_times < submission_time)),
        "check_event_timestamps_increasing": bool(np.all(event_times[1:] > event_times[:-1])),
        "mid_return_1_50_bps": 10_000 * (mid[-1] - mid[0]) / mid[0],
        "best_bid_change_count": int(np.count_nonzero(np.diff(bid_l1))),
        "best_ask_change_count": int(np.count_nonzero(np.diff(ask_l1))),
        "net_best_quote_direction_ticks": float(
            np.round(((bid_l1[-1] - bid_l1[0]) + (ask_l1[-1] - ask_l1[0])) / TICK_SIZE)
        ),
        "spread_mean_50": float(spread.mean()),
        "spread_max_50": float(spread.max()),
        "spread_final": float(spread[-1]),
        "spread_one_tick_fraction": float(np.mean(spread == 1)),
        "imbalance_mean_50": float(imbalance.mean()),
        "imbalance_std_50": float(imbalance.std(ddof=0)),
        "imbalance_min_50": float(imbalance.min()),
        "imbalance_max_50": float(imbalance.max()),
        "imbalance_final": float(imbalance[-1]),
        "imbalance_trend_50": linear_trend(imbalance),
        "bid_depth_change_1_50": float(bid_depth[-1] - bid_depth[0]),
        "ask_depth_change_1_50": float(ask_depth[-1] - ask_depth[0]),
        "micro_gap_final_bps": float(micro_gap_bps[-1]),
        "micro_gap_mean_50_bps": float(micro_gap_bps.mean()),
        "micro_gap_trend_50": linear_trend(micro_gap_bps),
        "lookback_span_ms": float(offsets_ms[-1] - offsets_ms[0]),
        "mean_inter_event_ms": float(np.diff(offsets_ms).mean()),
        "last_event_to_submission_ms": float(-offsets_ms[-1]),
    }

    for event_type in EVENT_TYPES:
        row[f"event_count_{event_type}"] = int(event_type_counts[event_type])
        for event_side in EVENT_SIDES:
            row[f"event_count_{event_type}_{event_side}"] = int(
                event_type_side_counts[(event_type, event_side)]
            )

    return row

In [4]:
started = time.perf_counter()
history_rows = []

with open_json_binary(JSON_PATH) as file:
    for order in ijson.items(file, "orders.item"):
        history_rows.append(summarize_history(order))

history_features = pd.DataFrame(history_rows)
elapsed_seconds = time.perf_counter() - started

print("History feature table shape:", history_features.shape)
print("Elapsed seconds:", round(elapsed_seconds, 2))
history_features.head(3)

History feature table shape: (10000, 41)
Elapsed seconds: 7.19


,order_id,history_snapshot_count,check_sequence_1_to_50,check_offsets_strictly_negative,check_offsets_strictly_increasing,check_event_timestamps_before_submission,check_event_timestamps_increasing,mid_return_1_50_bps,best_bid_change_count,best_ask_change_count,...,event_count_limit_submission_ask,event_count_cancellation,event_count_cancellation_bid,event_count_cancellation_ask,event_count_market_trade,event_count_market_trade_bid,event_count_market_trade_ask,event_count_modification,event_count_modification_bid,event_count_modification_ask
0,ORD000001,50,True,True,True,True,True,-4.879239,13,17,...,9,12,5,7,10,6,4,7,3,4
1,ORD000002,50,True,True,True,True,True,-2.089646,7,9,...,12,17,9,8,6,3,3,3,2,1
2,ORD000003,50,True,True,True,True,True,-0.348408,7,10,...,14,15,7,8,6,2,4,4,2,2


In [5]:
check_columns = [
    column for column in history_features.columns
    if column.startswith("check_")
]

event_count_columns = [
    f"event_count_{event_type}" for event_type in EVENT_TYPES
]

validation_summary = pd.Series({
    "history_rows": len(history_features),
    "unique_history_order_ids": history_features["order_id"].nunique(),
    "duplicate_history_order_ids": history_features["order_id"].duplicated().sum(),
    "orders_with_50_snapshots": history_features["history_snapshot_count"].eq(50).sum(),
    "all_sequence_checks_pass": history_features["check_sequence_1_to_50"].all(),
    "all_offsets_negative": history_features["check_offsets_strictly_negative"].all(),
    "all_offsets_increasing": history_features["check_offsets_strictly_increasing"].all(),
    "all_event_times_before_submission": history_features[
        "check_event_timestamps_before_submission"
    ].all(),
    "all_event_times_increasing": history_features[
        "check_event_timestamps_increasing"
    ].all(),
    "orders_with_50_counted_events": history_features[event_count_columns]
        .sum(axis=1)
        .eq(50)
        .sum(),
})

assert validation_summary["history_rows"] == 10_000
assert validation_summary["unique_history_order_ids"] == 10_000
assert validation_summary["duplicate_history_order_ids"] == 0
assert validation_summary["orders_with_50_snapshots"] == 10_000
assert validation_summary["all_sequence_checks_pass"]
assert validation_summary["all_offsets_negative"]
assert validation_summary["all_offsets_increasing"]
assert validation_summary["all_event_times_before_submission"]
assert validation_summary["all_event_times_increasing"]
assert validation_summary["orders_with_50_counted_events"] == 10_000

excel_ids = set(orders["order_id"].astype(str))
history_ids = set(history_features["order_id"])
assert excel_ids == history_ids

validation_summary.to_frame("value")

,value
history_rows,10000
unique_history_order_ids,10000
duplicate_history_order_ids,0
orders_with_50_snapshots,10000
all_sequence_checks_pass,True
all_offsets_negative,True
all_offsets_increasing,True
all_event_times_before_submission,True
all_event_times_increasing,True
orders_with_50_counted_events,10000


## Order-side orientation

Raw JSON imbalance is bid-minus-ask oriented. To compare buy and sell orders consistently, the notebook defines positive oriented imbalance as greater depth on the studied order's own side:

- buy order: keep the bid-minus-ask sign;
- sell order: reverse the sign.

The same convention is used for net quote direction, as specified in the feature reference. Bid/ask depth changes and event counts are also converted into same-side and opposite-side variables. Raw bid/ask variables remain available for auditability.

In [6]:
model_data = orders.merge(
    history_features,
    on="order_id",
    how="left",
    validate="one_to_one",
)

buy_mask = model_data["side"].astype(str).str.lower().eq("buy")
side_sign = np.where(buy_mask, 1.0, -1.0)

model_data["oriented_net_best_quote_direction_ticks"] = (
    model_data["net_best_quote_direction_ticks"] * side_sign
)
model_data["oriented_imbalance_mean_50"] = (
    model_data["imbalance_mean_50"] * side_sign
)
model_data["oriented_imbalance_final"] = (
    model_data["imbalance_final"] * side_sign
)
model_data["oriented_imbalance_trend_50"] = (
    model_data["imbalance_trend_50"] * side_sign
)
model_data["oriented_imbalance_min_50"] = np.where(
    buy_mask,
    model_data["imbalance_min_50"],
    -model_data["imbalance_max_50"],
)
model_data["oriented_imbalance_max_50"] = np.where(
    buy_mask,
    model_data["imbalance_max_50"],
    -model_data["imbalance_min_50"],
)

model_data["same_side_depth_change_1_50"] = np.where(
    buy_mask,
    model_data["bid_depth_change_1_50"],
    model_data["ask_depth_change_1_50"],
)
model_data["opposite_side_depth_change_1_50"] = np.where(
    buy_mask,
    model_data["ask_depth_change_1_50"],
    model_data["bid_depth_change_1_50"],
)

for event_type in EVENT_TYPES:
    model_data[f"event_count_{event_type}_same_side"] = np.where(
        buy_mask,
        model_data[f"event_count_{event_type}_bid"],
        model_data[f"event_count_{event_type}_ask"],
    )
    model_data[f"event_count_{event_type}_opposite_side"] = np.where(
        buy_mask,
        model_data[f"event_count_{event_type}_ask"],
        model_data[f"event_count_{event_type}_bid"],
    )

assert model_data[history_features.columns.difference(["order_id"])].notna().all().all()
print("Merged model table shape:", model_data.shape)
print("History rows joined without missing values:", len(model_data))

Merged model table shape: (10000, 128)
History rows joined without missing values: 10000


In [7]:
feature_groups = {
    "price_movement": [
        "mid_return_1_50_bps",
        "best_bid_change_count",
        "best_ask_change_count",
        "oriented_net_best_quote_direction_ticks",
    ],
    "spread_behavior": [
        "spread_mean_50",
        "spread_max_50",
        "spread_final",
        "spread_one_tick_fraction",
    ],
    "imbalance_behavior": [
        "oriented_imbalance_mean_50",
        "imbalance_std_50",
        "oriented_imbalance_min_50",
        "oriented_imbalance_max_50",
        "oriented_imbalance_final",
        "oriented_imbalance_trend_50",
    ],
    "depth_change": [
        "same_side_depth_change_1_50",
        "opposite_side_depth_change_1_50",
    ],
    "microprice_signal": [
        "micro_gap_final_bps",
        "micro_gap_mean_50_bps",
        "micro_gap_trend_50",
    ],
    "event_composition": [
        f"event_count_{event_type}_{orientation}"
        for event_type in EVENT_TYPES
        for orientation in ("same_side", "opposite_side")
    ],
    "event_timing": [
        "lookback_span_ms",
        "mean_inter_event_ms",
        "last_event_to_submission_ms",
    ],
}

history_constructed_features = [
    feature
    for features in feature_groups.values()
    for feature in features
]

feature_inventory = pd.DataFrame([
    {
        "feature_group": group,
        "feature_count": len(features),
        "example_features": ", ".join(features[:3]),
    }
    for group, features in feature_groups.items()
])

assert len(history_constructed_features) == len(set(history_constructed_features))
assert set(history_constructed_features).issubset(model_data.columns)

print("Constructed history features:", len(history_constructed_features))
feature_inventory

Constructed history features: 30


,feature_group,feature_count,example_features
0,price_movement,4,"mid_return_1_50_bps, best_bid_change_count, be..."
1,spread_behavior,4,"spread_mean_50, spread_max_50, spread_final"
2,imbalance_behavior,6,"oriented_imbalance_mean_50, imbalance_std_50, ..."
3,depth_change,2,"same_side_depth_change_1_50, opposite_side_dep..."
4,microprice_signal,3,"micro_gap_final_bps, micro_gap_mean_50_bps, mi..."
5,event_composition,8,"event_count_limit_submission_same_side, event_..."
6,event_timing,3,"lookback_span_ms, mean_inter_event_ms, last_ev..."


In [8]:
# Feature-quality inspection uses train and validation only.
development = model_data.loc[
    model_data["data_split"].astype(str).str.lower().isin(["train", "validation"])
].copy()

quality_rows = []
for feature in history_constructed_features:
    values = pd.to_numeric(development[feature], errors="coerce")
    quality_rows.append({
        "feature": feature,
        "missing_values": int(values.isna().sum()),
        "nonfinite_values": int((~np.isfinite(values)).sum()),
        "unique_values": int(values.nunique()),
        "train_mean": model_data.loc[
            model_data["data_split"].astype(str).str.lower().eq("train"), feature
        ].mean(),
        "validation_mean": model_data.loc[
            model_data["data_split"].astype(str).str.lower().eq("validation"), feature
        ].mean(),
    })

feature_quality = pd.DataFrame(quality_rows)

assert feature_quality["missing_values"].eq(0).all()
assert feature_quality["nonfinite_values"].eq(0).all()

constant_features = feature_quality.loc[
    feature_quality["unique_values"].le(1), "feature"
].tolist()
assert constant_features == ["last_event_to_submission_ms"]

history_model_features = [
    feature
    for feature in history_constructed_features
    if feature not in constant_features
]

print("Development rows inspected:", len(development))
print("Test target columns inspected:", 0)
print("Constant audit-only features:", constant_features)
print("Model-ready history features:", len(history_model_features))
feature_quality.head(12).round(4)

Development rows inspected: 8500
Test target columns inspected: 0
Constant audit-only features: ['last_event_to_submission_ms']
Model-ready history features: 29


,feature,missing_values,nonfinite_values,unique_values,train_mean,validation_mean
0,mid_return_1_50_bps,0,0,6088,0.0298,-0.0119
1,best_bid_change_count,0,0,22,9.3254,9.2713
2,best_ask_change_count,0,0,25,11.8463,11.6473
3,oriented_net_best_quote_direction_ticks,0,0,62,-0.0001,-0.0373
4,spread_mean_50,0,0,121,1.5926,1.5966
5,spread_max_50,0,0,4,2.7634,2.7207
6,spread_final,0,0,4,1.6031,1.6053
7,spread_one_tick_fraction,0,0,51,0.5700,0.5664
8,oriented_imbalance_mean_50,0,0,8499,0.0010,-0.0032
9,imbalance_std_50,0,0,8500,0.0376,0.0380


## Checkpoint findings

- The Excel source contains 10,000 unique AAPL orders with the required chronological split.
- The JSON source contains one matching history per order and exactly 50 snapshots per history.
- Sequence steps, negative offsets, event ordering, and the strict pre-submission timestamp rule all pass for every order.
- Seven required feature families are now represented by a compact model-ready history feature set.
- `last_event_to_submission_ms` is retained as an audit feature but excluded from modeling because it is always 1 ms.
- No test outcomes were inspected, and no raw or processed data files were written.

### Next notebook

Define feature sets A–D, fit the empirical baseline, Logistic Regression, small Decision Tree, and Random Forest on training data, and use validation data for model selection and probability calibration. The test split should remain sealed until the analysis choices are fixed.